#Scrapping Speeches from the Fed

The following chunk looks to scrape all speeches from the federal reserve site from 2020 to 2026.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_all_fed_speeches(start_year=2020, end_year=2026):
    print(f"1. Scraping Federal Reserve Speeches from {start_year} to {end_year}...")

    base_url = "https://www.federalreserve.gov"
    speech_data = []

    # Loop through each years archive page
    for year in range(start_year, end_year + 1):
        print(f"   -> Fetching links for {year}...")
        archive_url = f"{base_url}/newsevents/speech/{year}-speeches.htm"

        response = requests.get(archive_url)
        if response.status_code != 200:
            print(f"      [!] Could not access {year} archive (It might not exist yet).")
            continue

        soup = BeautifulSoup(response.text, 'html.parser')

        speech_links = []
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']
            # Excluding archive link
            if '/newsevents/speech/' in href and href.endswith('.htm') and f'{year}-speeches' not in href:
                if href.startswith('/'):
                    speech_links.append(base_url + href)
                else:
                    speech_links.append(href)

        speech_links = list(set(speech_links))
        print(f"      Found {len(speech_links)} speeches for {year}. Extracting text...")

        # Now visit each specific speech page
        for speech_url in speech_links:
            try:
                speech_response = requests.get(speech_url)
                speech_soup = BeautifulSoup(speech_response.text, 'html.parser')

                # Found html classes
                date_tag = speech_soup.find('p', class_='article__time')
                title_tag = speech_soup.find('h3', class_='title')
                speaker_tag = speech_soup.find('p', class_='speaker')

                # Body text
                article_div = speech_soup.find('div', id='article')

                speech_date = date_tag.text.strip() if date_tag else "Date Not Found"
                speech_title = title_tag.text.strip() if title_tag else "Title Not Found"
                speech_speaker = speaker_tag.text.strip() if speaker_tag else "Speaker Not Found"
                speech_text = article_div.get_text(separator=' ', strip=True) if article_div else "Text Not Found"

                # Only add to our list if we actually found text
                if speech_text != "Text Not Found":
                    speech_data.append({
                        'Date': speech_date,
                        'Speaker': speech_speaker,
                        'Title': speech_title,
                        'Text': speech_text,
                        'URL': speech_url
                    })

                # Pause for no blocking hopefully
                time.sleep(1)

            except Exception as e:
                print(f"      [!] Error fetching {speech_url}: {e}")

    df = pd.DataFrame(speech_data)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df = df.sort_values(by='Date')

    print(f"\n2. Successfully scraped {len(df)} total speeches!")
    print("3. Saving to 'all_fed_speeches.csv'...")

    df.to_csv('all_fed_speeches.csv', index=False, encoding='utf-8')
    print("Done.")

    return df

all_speeches_df = scrape_all_fed_speeches(start_year=2020, end_year=2026)

1. Scraping Federal Reserve Speeches from 2020 to 2026...
   -> Fetching links for 2020...
      Found 53 speeches for 2020. Extracting text...
   -> Fetching links for 2021...
      Found 68 speeches for 2021. Extracting text...
   -> Fetching links for 2022...
      Found 46 speeches for 2022. Extracting text...
   -> Fetching links for 2023...
      Found 95 speeches for 2023. Extracting text...
   -> Fetching links for 2024...
      Found 104 speeches for 2024. Extracting text...
   -> Fetching links for 2025...
      Found 117 speeches for 2025. Extracting text...
   -> Fetching links for 2026...
      Found 13 speeches for 2026. Extracting text...

2. Successfully scraped 496 total speeches!
3. Saving to 'all_fed_speeches.csv'...
Done.


In [ ]:
fed = pd.read_csv("all_fed_speeches.csv")
fed.head()

,Date,Speaker,Title,Text,URL
0,2020-01-08,Governor Lael Brainard,Strengthening the Community Reinvestment Act b...,"January 08, 2020 Strengthening the Community R...",https://www.federalreserve.gov/newsevents/spee...
1,2020-01-09,Vice Chair Richard H. Clarida,U.S. Economic Outlook and Monetary Policy,"January 09, 2020 U.S. Economic Outlook and Mon...",https://www.federalreserve.gov/newsevents/spee...
2,2020-01-16,Governor Michelle W. Bowman,The Outlook for Housing,"January 16, 2020 The Outlook for Housing Gover...",https://www.federalreserve.gov/newsevents/spee...
3,2020-01-17,Vice Chair for Supervision Randal K. Quarles,"Spontaneity and Order: Transparency, Accountab...","January 17, 2020 Spontaneity and Order: Transp...",https://www.federalreserve.gov/newsevents/spee...
4,2020-02-05,Governor Lael Brainard,The Digitalization of Payments and Currency: S...,"February 05, 2020 The Digitalization of Paymen...",https://www.federalreserve.gov/newsevents/spee...


#Scrapping Press Releases from FR

This chunk is dedicated to press releases from the Federal Reserve site

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_all_fed_press_releases(start_year=2020, end_year=2026):
    print(f"1. Scraping Federal Reserve Press Releases from {start_year} to {end_year}...")

    base_url = "https://www.federalreserve.gov"
    press_data = []

    # Loop through each years archive page
    for year in range(start_year, end_year + 1):
        print(f"   -> Fetching links for {year}...")
        archive_url = f"{base_url}/newsevents/pressreleases/{year}-press.htm"
        response = requests.get(archive_url)

        # Trying alternatives to their site's naming
        if response.status_code != 200:
            alt_url = f"{base_url}/newsevents/pressreleases/{year}press.htm"
            response = requests.get(alt_url)

        if response.status_code != 200:
            print(f"      [!] Could not access {year} archive. Skipping...")
            continue

        soup = BeautifulSoup(response.text, 'html.parser')

        press_links = []
        for a_tag in soup.find_all('a', href=True):
            href = a_tag['href']

            # Filter for press release links but exclude the archive pages themselves
            if '/newsevents/pressreleases/' in href and href.endswith('.htm') and 'press.htm' not in href:
                if href.startswith('/'):
                    press_links.append(base_url + href)
                else:
                    press_links.append(href)

        # Remove duplicate links
        press_links = list(set(press_links))
        print(f"      Found {len(press_links)} press releases for {year}. Extracting text...")

        for press_url in press_links:
            try:
                press_response = requests.get(press_url)
                press_soup = BeautifulSoup(press_response.text, 'html.parser')

                # Using HTML classes that I found in the HTML
                date_tag = press_soup.find('p', class_='article__time')
                title_tag = press_soup.find('h3', class_='title')
                release_time_tag = press_soup.find('p', class_='releaseTime')

                article_div = press_soup.find('div', id='article')

                # Alternatives if it fails
                press_date = date_tag.text.strip() if date_tag else "Date Not Found"
                press_title = title_tag.text.strip() if title_tag else "Title Not Found"
                press_release_time = release_time_tag.text.strip() if release_time_tag else "Release Time Not Found"
                press_text = article_div.get_text(separator=' ', strip=True) if article_div else "Text Not Found"

                if press_text != "Text Not Found":
                    press_data.append({
                        'Date': press_date,
                        'Release_Time': press_release_time,
                        'Title': press_title,
                        'Text': press_text,
                        'URL': press_url
                    })

                # Pause for good practices
                time.sleep(0.3)

            except Exception as e:
                print(f"      [!] Error fetching {press_url}: {e}")

    df = pd.DataFrame(press_data)

    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

    df = df.sort_values(by='Date')

    print(f"\n2. Successfully scraped {len(df)} total press releases!")
    print("3. Saving to 'all_fed_press_releases.csv'...")

    df.to_csv('all_fed_press_releases.csv', index=False, encoding='utf-8')
    print("Done")

    return df

# Run the function
all_press_releases_df = scrape_all_fed_press_releases(start_year=2020, end_year=2026)

1. Scraping Federal Reserve Press Releases from 2020 to 2026...
   -> Fetching links for 2020...
      Found 220 press releases for 2020. Extracting text...
   -> Fetching links for 2021...
      Found 145 press releases for 2021. Extracting text...
   -> Fetching links for 2022...
      Found 147 press releases for 2022. Extracting text...
   -> Fetching links for 2023...
      Found 127 press releases for 2023. Extracting text...
   -> Fetching links for 2024...
      Found 120 press releases for 2024. Extracting text...
   -> Fetching links for 2025...
      Found 135 press releases for 2025. Extracting text...
   -> Fetching links for 2026...
      Found 14 press releases for 2026. Extracting text...

2. Successfully scraped 908 total press releases!
3. Saving to 'all_fed_press_releases.csv'...
Done! Check your folder for the file.


#Statements and Minutes from FOMC

Statements and Minutes not from the Fed Reserve but from the FedTools API

In [ ]:
import pandas as pd
from FedTools import MonetaryPolicyCommittee, FederalReserveMins

def get_historical_fed_docs():
    print("Scraping Federal Reserve FOMC statements and minutes since 2020...")

    print("   -> Fetching Statements...")
    monetary_policy = MonetaryPolicyCommittee(
        main_url = 'https://www.federalreserve.gov',
        calendar_url = 'https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm',
        historical_split = 2020,
        verbose = True,
        thread_num = 10
    )
    statements_df = monetary_policy.find_statements()

    statements_df.columns = ['Text']
    statements_df['Document_Type'] = 'Statement'

    print("   -> Fetching Minutes...")
    reserve_mins = FederalReserveMins(
        main_url = 'https://www.federalreserve.gov',
        calendar_url = 'https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm',
        historical_split = 2020,
        verbose = True,
        thread_num = 10
    )
    minutes_df = reserve_mins.find_minutes()

    minutes_df.columns = ['Text']
    minutes_df['Document_Type'] = 'Minutes'

    print("   -> Combining datasets...")
    combined_df = pd.concat([statements_df, minutes_df]).reset_index()

    if 'index' in combined_df.columns:
        combined_df = combined_df.rename(columns={'index': 'Date'})

    combined_df.to_csv("fed_statements_and_minutes_since_2020.csv", index=False, encoding='utf-8')
    print("\nDone")

    return combined_df

# Run the function
historical_data = get_historical_fed_docs()
print(historical_data.head())

Scraping Federal Reserve FOMC statements and minutes since 2020...
   -> Fetching Statements...
Constructing links between 1994 and 2026
Extracting the past 242 FOMC Statements.
Retrieving articles.
..................................................................................................................................................................................................................................................   -> Fetching Minutes...
Constructing links between 1995 and 2026
Extracting Federal Reserve Minutes.
Retrieving articles.
...............................................................................................................................................................................................................................................................   -> Combining datasets...

Success! Saved to fed_statements_and_minutes_since_2020.csv
        Date                                               Text Document_Type
0 1994-02-04 

In [ ]:
historical_data['Date'] = pd.to_datetime(historical_data['Date'])
yearly_counts = pd.crosstab(historical_data['Date'].dt.year, historical_data['Document_Type'])

print(yearly_counts)

Document_Type  Minutes  Statement
Date                             
1994                 0          6
1995                 8          3
1996                 8          1
1997                 8          1
1998                 8          3
1999                 8          6
2000                 8          8
2001                 8         11
2002                 8          8
2003                 8          8
2004                 8          8
2005                 8          8
2006                 8          8
2007                 8         10
2008                 7         11
2009                 8          8
2010                 8          9
2011                 8          8
2012                 8          8
2013                 8          8
2014                 8          8
2015                 8          8
2016                 8          8
2017                 8          8
2018                 8          8
2019                 8          9
2020                 8         10
2021          